# Tester la detection par expressions regulieres

Ce notebook ne lit pas de PDF. Il teste une branche locale basee uniquement sur des expressions regulieres, sans Presidio et sans modele NLP.

`test_cases` contient seulement trois colonnes : `text`, `control_family`, `comment`.

Seule la colonne `text` est envoyee au traitement. `control_family` et `comment` servent uniquement a lire, filtrer et commenter les resultats dans le notebook.

Les chemins `D:\Workspaces\modelStore` et `D:\Workspaces\ModelCache` restent affiches dans les parametres pour garder la meme convention que les autres notebooks, mais cette branche ne telecharge rien et ne charge aucun modele.


In [ ]:
from pathlib import Path
import os
import re
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_colwidth", 180)


In [ ]:
BRANCH_NAME = "regex"
BRANCH_SCORE_COLUMN = "regex_score"

MODEL_CACHE_DIR = r"D:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"D:\Workspaces\modelStore"

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = rf"{MODEL_CACHE_DIR}\transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = rf"{MODEL_CACHE_DIR}\hub"
os.environ["COMPLIANCE_NLP_MODEL_CACHE"] = MODEL_CACHE_DIR
os.environ["COMPLIANCE_NLP_MODEL_STORE"] = MODEL_STORE_DIR

REGEX_ENTITIES = [
    "EMAIL_ADDRESS",
    "PHONE_NUMBER_FR",
    "IBAN_FR",
    "FR_NIR",
]

params = {
    "branch": BRANCH_NAME,
    "model_cache_dir": MODEL_CACHE_DIR,
    "model_store_dir": MODEL_STORE_DIR,
    "regex_entities": REGEX_ENTITIES,
    "downloads": "none",
}
params


In [ ]:
REGEX_RULES = [
    {
        "entity_type": "EMAIL_ADDRESS",
        "pattern": re.compile(r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"),
        "score": 0.95,
    },
    {
        "entity_type": "PHONE_NUMBER_FR",
        "pattern": re.compile(r"(?<!\d)(?:\+33[\s.-]?|0)[1-9](?:[\s.-]?\d{2}){4}(?!\d)"),
        "score": 0.90,
    },
    {
        "entity_type": "IBAN_FR",
        "pattern": re.compile(r"(?i)\bFR\d{2}(?:[\s-]?[A-Z0-9]{4}){5}[\s-]?[A-Z0-9]{3}\b"),
        "score": 0.95,
    },
    {
        "entity_type": "FR_NIR",
        "pattern": re.compile(
            r"(?i)\b[12]\s?\d{2}\s?(?:0[1-9]|1[0-2]|20)\s?"
            r"(?:\d{2}|2A|2B|97[1-6]|98[4678])\s?\d{3}\s?\d{3}\s?\d{2}\b"
        ),
        "score": 0.95,
    },
]


def find_regex_entities(text: str) -> list[dict]:
    findings = []
    for rule in REGEX_RULES:
        for match in rule["pattern"].finditer(text):
            findings.append(
                {
                    "entity_type": rule["entity_type"],
                    "matched_text": match.group(0),
                    "start": match.start(),
                    "end": match.end(),
                    "regex_score": rule["score"],
                }
            )
    return sorted(findings, key=lambda item: (item["start"], item["end"], item["entity_type"]))


In [ ]:
TEST_CASES_PATH = ROOT / "configs" / "test_cases.csv"
TEST_CASE_COLUMNS = ["text", "control_family", "comment"]

test_cases = pd.read_csv(TEST_CASES_PATH, keep_default_na=False)
missing_columns = set(TEST_CASE_COLUMNS) - set(test_cases.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans {TEST_CASES_PATH}: {sorted(missing_columns)}")

test_cases = test_cases[TEST_CASE_COLUMNS]
test_cases


In [ ]:
def result_rows_for_case(case_index: int, case: pd.Series) -> list[dict]:
    text = case["text"]
    findings = find_regex_entities(text)
    if not findings:
        return [
            {
                "case_index": case_index,
                "control_family": case["control_family"],
                "comment": case["comment"],
                "text": text.strip(),
                "detection_engine": BRANCH_NAME,
                "entity_type": None,
                "matched_text": None,
                "start": None,
                "end": None,
                "regex_score": None,
            }
        ]

    rows = []
    for finding in findings:
        rows.append(
            {
                "case_index": case_index,
                "control_family": case["control_family"],
                "comment": case["comment"],
                "text": text.strip(),
                "detection_engine": BRANCH_NAME,
                "entity_type": finding["entity_type"],
                "matched_text": finding["matched_text"],
                "start": finding["start"],
                "end": finding["end"],
                "regex_score": finding["regex_score"],
            }
        )
    return rows


In [ ]:
rows = []
for case_index, case in test_cases.iterrows():
    rows.extend(result_rows_for_case(case_index, case))

results_df = pd.DataFrame(rows)
results_df


In [ ]:
alerts_df = results_df[results_df["entity_type"].notna()].copy()
alerts_df.sort_values(["case_index", "entity_type", "start"])


In [ ]:
summary_df = (
    results_df.assign(has_detection=results_df["entity_type"].notna())
    .groupby(["case_index", "control_family", "comment"], dropna=False)
    .agg(
        detection_count=("has_detection", "sum"),
        max_score=("regex_score", "max"),
        entities=("entity_type", lambda values: sorted({value for value in values if pd.notna(value)})),
    )
    .reset_index()
)
summary_df


In [ ]:
entity_summary_df = (
    alerts_df.groupby(["entity_type"], dropna=False)
    .agg(
        detection_count=("entity_type", "size"),
        max_score=("regex_score", "max"),
    )
    .reset_index()
    .sort_values(["entity_type"])
)
entity_summary_df
